# Hito1 Notebook

IIT414W Capstone NextLap · reference reproduction, leakage checks and OLS comparison

Part A reproduces the three references (training median, persistence, OLS) on the common validation rows. Part B builds the candidate feature `recent_std_s` and checks that it uses nothing after lap t (leakage checks L1–L4). Part C compares reference OLS vs OLS + `recent_std_s` on identical `pair_id` values (checks C1–C4).

The course helper `nextlap_reference_v1.py` (released in `IIT414W_Capstone_NextLap_StudentData_v1.zip`) is reused with attribution for `make_model`, `load_snapshots` and the feature constants.

Set `DATA_DIR` in the next cell to the folder holding the extracted student data, then run all cells top to bottom.

In [1]:
from pathlib import Path

DATA_DIR = Path('.')        # folder with train_clean_v1.csv, validation_clean_v1.csv, data/, nextlap_reference_v1.py
OUT_DIR = Path('outputs')   # generated tables are written here
RANDOM_SEED = 414

In [2]:
import importlib.util
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

spec = importlib.util.spec_from_file_location('nextlap_reference_v1', DATA_DIR / 'nextlap_reference_v1.py')
ref = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ref)

TARGET = ref.TARGET
pd.options.display.float_format = '{:.6f}'.format
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}')
print('numeric:', ref.NUMERIC)
print('categorical:', ref.CATEGORICAL)
print('target:', TARGET)

pandas 3.0.5 | numpy 2.5.3 | scikit-learn 1.9.1
numeric: ['current_lap_s', 'recent_mean_s', 'current_lap_number', 'current_tyre_life']
categorical: ['current_compound', 'current_team']
target: next_lap_s


## Part A · Reproduced reference

### A1. Load the fixed split and check the comparison population

The cell stops with an `AssertionError` if the rows, rounds or `pair_id` values differ from the release.

In [3]:
kinds = {'driver_number': str}
train = pd.read_csv(DATA_DIR / 'train_clean_v1.csv', dtype=kinds)
validation = pd.read_csv(DATA_DIR / 'validation_clean_v1.csv', dtype=kinds)
features = ref.NUMERIC + ref.CATEGORICAL

assert len(train) == 5772 and len(validation) == 1808
assert set(train['round']) == {1, 2, 3, 4, 5, 6, 7} and set(validation['round']) == {8, 9}
assert train.pair_id.is_unique and validation.pair_id.is_unique
assert not set(train.pair_id) & set(validation.pair_id), 'train/validation pair_id overlap'
assert train.clean_pair.all() and validation.clean_pair.all()
assert not train[features + [TARGET]].isna().any().any()
assert not validation[features + [TARGET]].isna().any().any()

print(f'train rows {len(train)} (rounds 1-7) | validation rows {len(validation)} (rounds 8-9)')
print('unique pair_id in each split, zero overlap, no missing reference inputs')

train rows 5772 (rounds 1-7) | validation rows 1808 (rounds 8-9)
unique pair_id in each split, zero overlap, no missing reference inputs


### A2. Fit on training rows only, predict the validation rows

The imputer, scaler and one-hot encoder live inside the helper's pipeline, so `.fit()` on training rows is the only place they learn anything. The training median also comes from training rows only.

In [4]:
def mae_table(frame, predictions):
    """MAE in seconds with row counts: all validation rows, then each validation race."""
    rows = []
    for name, predicted in predictions.items():
        error = (frame[TARGET] - predicted).abs()
        rows.append({'model': name, 'population': 'validation rounds 8-9 (clean pairs)',
                     'rows': len(error), 'MAE_s': error.mean()})
        for (rnd, event), part in error.groupby([frame['round'], frame['event_name']]):
            rows.append({'model': name, 'population': f'round {rnd} {event} (clean pairs)',
                         'rows': len(part), 'MAE_s': part.mean()})
    return pd.DataFrame(rows)


ols_reference = ref.make_model('OLS').fit(train[features], train[TARGET])
reference_predictions = {
    'training_median': np.full(len(validation), train[TARGET].median()),
    'persistence': validation['current_lap_s'].to_numpy(),
    'ols_reference': ols_reference.predict(validation[features]),
}
reference_metrics = mae_table(validation, reference_predictions)
reference_metrics

,model,population,rows,MAE_s
0,training_median,validation rounds 8-9 (clean pairs),1808,11.376111
1,training_median,round 8 Austrian Grand Prix (clean pairs),1066,11.715781
2,training_median,round 9 British Grand Prix (clean pairs),742,10.888121
3,persistence,validation rounds 8-9 (clean pairs),1808,0.409289
4,persistence,round 8 Austrian Grand Prix (clean pairs),1066,0.400516
5,persistence,round 9 British Grand Prix (clean pairs),742,0.421892
6,ols_reference,validation rounds 8-9 (clean pairs),1808,0.386707
7,ols_reference,round 8 Austrian Grand Prix (clean pairs),1066,0.373571
8,ols_reference,round 9 British Grand Prix (clean pairs),742,0.405579


In [5]:
reference_metrics.to_csv(OUT_DIR / 'reference_metrics.csv', index=False, float_format='%.6f')
traced = validation[['pair_id', 'round', TARGET]].copy()
for name, predicted in reference_predictions.items():
    traced['pred_' + name] = predicted
traced.to_csv(OUT_DIR / 'reference_predictions.csv', index=False, float_format='%.6f')
print('wrote', OUT_DIR / 'reference_metrics.csv', 'and', OUT_DIR / 'reference_predictions.csv')

wrote outputs/reference_metrics.csv and outputs/reference_predictions.csv


**Reading the table.** OLS beats persistence by about 0.023 s and is lower in both races. The gap is small because the current lap already carries most of the information about the next one. The median is about 11 s off because one constant cannot follow lap-time differences between circuits. These figures describe retrospectively selected clean pairs, not every lap of a live race.

## Part B · Candidate feature and temporal availability

**Candidate.** `recent_std_s` is the sample standard deviation (ddof = 1) of the same clean lap times that feed the supplied `recent_mean_s`: at most three laps ending at lap t, inside one season/round/driver/stint, built on the original timeline before clean-pair selection. Fewer than two valid laps gives a missing value, filled by the training-fitted median imputer, so no row is dropped.

**Time rule.** The prediction is made at the end of lap t. Only laps numbered t or earlier, from the same driver, race and tyre stint, may feed a feature. Nothing from lap t+1, t+2, t+3 or later may be used: no `next_*` column, no target, no `clean_pair` flag (it is computed with t+1 information).

**Leakage checks declared before the comparison (each stops the notebook on failure):**

| Check | Inspects | Failure |
|---|---|---|
| L1 last source lap | For every row, the latest lap number and timestamp inside the rolling window. | A source lap later than t, or a window wider than three laps. |
| L2 truncation | For a seeded sample of validation rows, delete every lap after t and rebuild the feature. | The value changes, which would mean later laps influenced it. |
| L3 boundaries | The number of laps in the window never exceeds the clean laps seen so far in that stint. | A window that reaches into another stint, driver or race. |
| L4 model inputs | The feature lists passed to the models. | The target, `clean_pair` or any `next_*` column among the inputs. |

### B1. Build the feature on the original timeline

In [6]:
FEATURE = 'recent_std_s'
WINDOW = 3


def add_history(frame):
    """Rolling history ending at each lap, inside one season/round/driver/stint.

    Same clean-lap rule as the helper's build_pairs: a bad observed lap never enters
    the history. rolling() looks backwards only, so a row never sees a later lap.
    """
    laps = frame.sort_values(ref.KEYS + ['lap_number']).reset_index(drop=True).copy()
    clean = (laps.lap_number.gt(1) & laps.lap_time_s.gt(0) & laps.is_accurate & ~laps.pit_in
             & ~laps.pit_out & ~laps.deleted & laps.track_status.eq('1'))
    laps['history_value'] = laps.lap_time_s.where(clean)
    laps['source_lap'] = laps.lap_number.where(clean)
    laps['source_end_s'] = laps.end_time_s.where(clean)
    stint_keys = ref.KEYS + ['stint']

    def roll(column, how, min_periods=1):
        return laps.groupby(stint_keys, dropna=False, sort=False)[column].transform(
            lambda s: getattr(s.rolling(WINDOW, min_periods=min_periods), how)())

    laps['rebuilt_mean_s'] = roll('history_value', 'mean')
    laps['rebuilt_count'] = roll('history_value', 'count')
    laps[FEATURE] = roll('history_value', 'std', min_periods=2)
    laps['first_source_lap'] = roll('source_lap', 'min')
    laps['last_source_lap'] = roll('source_lap', 'max')
    laps['last_source_end_s'] = roll('source_end_s', 'max')
    laps['clean_so_far'] = laps.history_value.notna().groupby(
        [laps[k] for k in stint_keys], dropna=False, sort=False).cumsum()
    laps['pair_id'] = (laps.season.astype(str) + '-R' + laps['round'].astype(str) + '-'
                       + laps.driver_number + '-L' + laps.lap_number.astype(int).astype(str))
    assert laps.pair_id.is_unique
    return laps


raw, _ = ref.load_snapshots(DATA_DIR)   # verifies the SHA-256 of every race snapshot
laps = add_history(raw)                 # original timeline, before any clean-pair selection
history = laps[['pair_id', 'rebuilt_mean_s', 'rebuilt_count', FEATURE]]
print(f'{len(laps)} original laps, {int(laps[FEATURE].notna().sum())} with a {FEATURE} value')

10430 original laps, 8414 with a recent_std_s value


### B2. Leakage checks L1–L4

In [7]:
# L1: the newest lap in every window is lap t or earlier, and the window spans at most WINDOW laps.
used = laps[laps.rebuilt_count > 0]
assert (used.last_source_lap <= used.lap_number).all(), 'L1 failed: a source lap is later than t'
assert (used.last_source_end_s <= used.end_time_s).all(), 'L1 failed: a source timestamp is later than the end of t'
assert (used.first_source_lap >= used.lap_number - (WINDOW - 1)).all(), 'L1 failed: window wider than WINDOW laps'
print(f'L1 last source lap: passed on {len(used)} laps (latest source lap <= t, latest source timestamp <= end of t)')

# L2: delete everything after lap t and rebuild. If the value moved, later laps were leaking in.
sample_ids = validation.pair_id.sample(200, random_state=RANDOM_SEED)
full = laps.set_index('pair_id')
checked = 0
for pair_id in sample_ids:
    row = full.loc[pair_id]
    same_race = (raw.season == row.season) & (raw['round'] == row['round']) & (raw.driver_number == row.driver_number)
    past_only = add_history(raw[same_race & (raw.lap_number <= row.lap_number)]).set_index('pair_id').loc[pair_id]
    for column in [FEATURE, 'rebuilt_mean_s']:
        assert np.isclose(past_only[column], row[column], equal_nan=True), f'L2 failed at {pair_id}: {column}'
    checked += 1
print(f'L2 truncation: passed on {checked} sampled validation rows (laps t+1, t+2, ... removed, values unchanged)')

# L3: a window can never hold more laps than the clean laps seen so far in the same stint.
assert (laps.rebuilt_count <= laps.clean_so_far).all(), 'L3 failed: a window crosses a stint, driver or race boundary'
first_clean = laps[(laps.clean_so_far == 1) & laps.history_value.notna()]
assert first_clean.rebuilt_count.eq(1).all() and first_clean[FEATURE].isna().all(), 'L3 failed at a stint start'
print(f'L3 boundaries: passed ({len(first_clean)} stint starts restart the history with one lap and no {FEATURE})')

# L4: nothing known only after lap t is a model input.
candidate_inputs = ref.NUMERIC + [FEATURE] + ref.CATEGORICAL
forbidden = [name for name in candidate_inputs if name.startswith('next_') or name in (TARGET, 'clean_pair')]
assert not forbidden, f'L4 failed: {forbidden}'
print('L4 model inputs: passed ->', candidate_inputs)

L1 last source lap: passed on 9300 laps (latest source lap <= t, latest source timestamp <= end of t)


L2 truncation: passed on 200 sampled validation rows (laps t+1, t+2, ... removed, values unchanged)
L3 boundaries: passed (461 stint starts restart the history with one lap and no recent_std_s)
L4 model inputs: passed -> ['current_lap_s', 'recent_mean_s', 'current_lap_number', 'current_tyre_life', 'recent_std_s', 'current_compound', 'current_team']


A hand-inspectable stint boundary from the original timeline: the laps around the first pit stop of one driver in round 8. The window columns show which laps fed each value; after the stop the history starts again.

In [8]:
one = laps[(laps['round'] == 8) & (laps.driver_number == validation.driver_number.iloc[0])]
change = one.index[one.stint.diff().fillna(0).ne(0)][0]
one.loc[change - 4:change + 4, ['lap_number', 'stint', 'lap_time_s', 'pit_in', 'pit_out', 'first_source_lap',
                                 'last_source_lap', 'rebuilt_count', 'rebuilt_mean_s', FEATURE]]

,lap_number,stint,lap_time_s,pit_in,pit_out,first_source_lap,last_source_lap,rebuilt_count,rebuilt_mean_s,recent_std_s
7995,18.000000,1.000000,73.381000,False,False,16.000000,18.000000,3.000000,73.287000,0.116258
7996,19.000000,1.000000,74.330000,False,False,17.000000,19.000000,3.000000,73.622667,0.622723
7997,20.000000,1.000000,73.655000,False,False,18.000000,20.000000,3.000000,73.788667,0.488416
7998,21.000000,1.000000,78.354000,True,False,19.000000,20.000000,2.000000,73.992500,0.477297
7999,22.000000,2.000000,90.005000,False,True,NaN,NaN,0.000000,NaN,NaN
8000,23.000000,2.000000,72.148000,False,False,23.000000,23.000000,1.000000,72.148000,NaN
8001,24.000000,2.000000,78.463000,False,False,23.000000,23.000000,1.000000,72.148000,NaN
8002,25.000000,2.000000,96.995000,False,False,23.000000,23.000000,1.000000,72.148000,NaN
8003,26.000000,2.000000,84.972000,False,False,NaN,NaN,0.000000,NaN,NaN


**Conclusion of the availability step.** Every `recent_std_s` value is built from laps t−2 to t of the same stint, never from t+1 or later. Removing all later laps leaves the values unchanged, and the history restarts at every stint. The remaining future-dependence is in the *population*, not the feature: `clean_pair` selects rows using lap t+1, so results describe retrospectively clean pairs and cannot be used to route live predictions.

## Part C · Baseline vs OLS + `recent_std_s`

**Fixed:** target, training and validation `pair_id` rows, split, OLS, preprocessing protocol, MAE in seconds, seed 414.

**Comparison checks declared before the extension (each stops the notebook on failure):**

| Check | Inspects |
|---|---|
| C1 timeline | `recent_mean_s` and `recent_count` rebuilt from the raw snapshots equal the released columns on every row. |
| C2 rows | Validation `pair_id` set, count and order are unchanged after adding the feature. |
| C3 short history | `recent_std_s` is missing exactly where `recent_count < 2`. |
| C4 ablation | The augmented feature list minus `recent_std_s` reproduces the Part A OLS predictions. |

**Decision rule, fixed in advance.** Supported only if the augmented MAE is lower on the 1,808 rows, the paired difference is negative in both round 8 and round 9, and the 95% interval of the mean paired difference (bootstrap over whole driver-races, 2,000 draws, seed 414) lies entirely below zero. Anything else is reported as "no supported improvement" and kept in the trial log.

### C1. Attach the feature and run checks C1–C3

In [9]:
def attach(frame):
    merged = frame.merge(history, on='pair_id', how='left', validate='one_to_one')
    assert merged.pair_id.tolist() == frame.pair_id.tolist(), 'C2 failed: rows changed'
    assert np.allclose(merged.rebuilt_mean_s, merged.recent_mean_s, atol=1e-6), 'C1 failed: mean'
    assert merged.rebuilt_count.eq(merged.recent_count).all(), 'C1 failed: count'
    assert merged[FEATURE].isna().eq(merged.recent_count.lt(2)).all(), 'C3 failed'
    return merged


train_x, validation_x = attach(train), attach(validation)
assert set(validation_x.pair_id) == set(validation.pair_id) and len(validation_x) == 1808
print('C1 timeline: passed | C2 rows: passed (1,808 validation pair_id unchanged) | C3 short history: passed')
print(f'validation rows with {FEATURE} observed: {int(validation_x[FEATURE].notna().sum())}, '
      f'imputed: {int(validation_x[FEATURE].isna().sum())}')

C1 timeline: passed | C2 rows: passed (1,808 validation pair_id unchanged) | C3 short history: passed
validation rows with recent_std_s observed: 1660, imputed: 148


A hand-inspectable sample: one driver's first stint in round 8. Each `recent_std_s` value can be recomputed from the `current_lap_s` values on that row and the one or two rows before it.

In [10]:
sample = validation_x[(validation_x['round'] == 8) & (validation_x.driver_number == validation_x.driver_number.iloc[0])
                      & (validation_x.stint == 1)]
sample[['pair_id', 'stint', 'current_lap_s', 'recent_count', 'recent_mean_s', FEATURE]].head(6)

,pair_id,stint,current_lap_s,recent_count,recent_mean_s,recent_std_s
0,2026-R8-1-L2,1.000000,73.406000,1.000000,73.406000,NaN
1,2026-R8-1-L3,1.000000,73.015000,2.000000,73.210500,0.276479
2,2026-R8-1-L4,1.000000,72.630000,3.000000,73.017000,0.388004
3,2026-R8-1-L5,1.000000,72.439000,3.000000,72.694667,0.293395
4,2026-R8-1-L6,1.000000,72.493000,3.000000,72.520667,0.098460
5,2026-R8-1-L7,1.000000,73.459000,3.000000,72.797000,0.573944


### C2. Fit both models and run the ablation check C4

In [11]:
augmented_numeric = ref.NUMERIC + [FEATURE]
ablated_numeric = [name for name in augmented_numeric if name != FEATURE]


def fit_predict(numeric):
    columns = numeric + ref.CATEGORICAL
    model = ref.make_model('OLS', numeric=numeric)
    return model.fit(train_x[columns], train_x[TARGET]).predict(validation_x[columns])


extension_predictions = {'ols_reference_ablation': fit_predict(ablated_numeric),
                         'ols_plus_' + FEATURE: fit_predict(augmented_numeric)}
assert np.allclose(reference_predictions['ols_reference'], extension_predictions['ols_reference_ablation'], atol=1e-9), 'C4 failed'
print('C4 ablation reproduces the Part A OLS reference: passed')

extension_metrics = mae_table(validation_x, extension_predictions)
extension_metrics

C4 ablation reproduces the Part A OLS reference: passed


,model,population,rows,MAE_s
0,ols_reference_ablation,validation rounds 8-9 (clean pairs),1808,0.386707
1,ols_reference_ablation,round 8 Austrian Grand Prix (clean pairs),1066,0.373571
2,ols_reference_ablation,round 9 British Grand Prix (clean pairs),742,0.405579
3,ols_plus_recent_std_s,validation rounds 8-9 (clean pairs),1808,0.365223
4,ols_plus_recent_std_s,round 8 Austrian Grand Prix (clean pairs),1066,0.357604
5,ols_plus_recent_std_s,round 9 British Grand Prix (clean pairs),742,0.376169


### C3. Paired difference, interval and the declared rule

In [12]:
error = {name: (validation_x[TARGET] - p).abs() for name, p in extension_predictions.items()}
difference = error['ols_plus_' + FEATURE] - error['ols_reference_ablation']   # negative = augmented is better

groups = [part.to_numpy() for _, part in difference.groupby([validation_x['round'], validation_x['driver_number']])]
rng = np.random.default_rng(RANDOM_SEED)
means = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(2000)]
low, high = np.percentile(means, [2.5, 97.5])

by_round = difference.groupby(validation_x['round']).mean()
observed = validation_x[FEATURE].notna()
supported = bool(difference.mean() < 0 and high < 0 and (by_round < 0).all())

print(f'paired difference (augmented - reference): {difference.mean():+.6f} s on {len(difference)} rows')
print(f'95% cluster bootstrap interval: [{low:+.6f}, {high:+.6f}] s over {len(groups)} driver-races')
for rnd, value in by_round.items():
    print(f'  round {rnd}: {value:+.6f} s on {int((validation_x["round"] == rnd).sum())} rows')
print(f'  feature observed: {difference[observed].mean():+.6f} s on {int(observed.sum())} rows')
print(f'  feature imputed (short history): {difference[~observed].mean():+.6f} s on {int((~observed).sum())} rows')
print('declared rule ->', 'SUPPORTED' if supported else 'NOT SUPPORTED (negative result retained)')

paired difference (augmented - reference): -0.021484 s on 1808 rows
95% cluster bootstrap interval: [-0.031503, -0.012860] s over 43 driver-races
  round 8: -0.015967 s on 1066 rows
  round 9: -0.029410 s on 742 rows
  feature observed: -0.025216 s on 1660 rows
  feature imputed (short history): +0.020375 s on 148 rows
declared rule -> SUPPORTED


### C4. Save the comparison and append the trial log

Every run appends one row to `trial_log.csv`, including unsuccessful ones.

In [13]:
extension_metrics.to_csv(OUT_DIR / 'extension_metrics.csv', index=False, float_format='%.6f')
traced = validation_x[['pair_id', 'round', TARGET, FEATURE]].copy()
for name, predicted in extension_predictions.items():
    traced['pred_' + name] = predicted
traced.to_csv(OUT_DIR / 'extension_predictions.csv', index=False, float_format='%.6f')

trial = pd.DataFrame([{
    'run_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'candidate': FEATURE, 'window': WINDOW, 'model': 'OLS', 'validation_rows': len(validation_x),
    'MAE_reference_s': error['ols_reference_ablation'].mean(),
    'MAE_augmented_s': error['ols_plus_' + FEATURE].mean(),
    'difference_s': difference.mean(), 'ci95_low_s': low, 'ci95_high_s': high,
    'difference_round8_s': by_round.loc[8], 'difference_round9_s': by_round.loc[9],
    'rows_feature_observed': int(observed.sum()), 'rows_feature_imputed': int((~observed).sum()),
    'hypothesis_supported': supported}])
log = OUT_DIR / 'trial_log.csv'
trial.to_csv(log, mode='a', header=not log.exists(), index=False, float_format='%.6f')
print('wrote extension_metrics.csv, extension_predictions.csv and appended', log)

wrote extension_metrics.csv, extension_predictions.csv and appended outputs/trial_log.csv


**Limit.** A lower MAE on clean pairs does not show a causal effect of pace variability, performance on all race laps, or when a car should pit. Contextual errors and SHAP belong to Hito 2.